# 🌲 Forest Vegetation Zone Classification
### CSE274 — Applied Machine Learning Project
**Topic:** Forest Tuning and Vegetation Zone  
**Dataset:** Forest Cover Type (UCI / Kaggle)  
**Model:** Random Forest Classifier  

---
**Vegetation Zones (Target Classes):**
1. Spruce/Fir
2. Lodgepole Pine
3. Ponderosa Pine
4. Cottonwood/Willow
5. Aspen
6. Douglas-fir
7. Krummholz

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install kaggle API
!pip install kaggle --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

print('✅ All libraries imported successfully!')

## 📥 Step 2 — Download Dataset from Kaggle
> **How to get kaggle.json:**
> 1. Go to kaggle.com → Your Profile → Settings → API → Create New Token
> 2. It downloads `kaggle.json` — upload it below when prompted

> **OR skip this and use the direct download below (no login needed)**

In [ ]:
# ── OPTION A: Kaggle API (upload your kaggle.json when prompted) ──
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d uciml/forest-cover-type-dataset --unzip

# ── OPTION B: Direct download (no login needed) ──
!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz
!gunzip -f covtype.data.gz
print('✅ Dataset downloaded!')

## 📊 Step 3 — Load & Explore Dataset

In [ ]:
# Column names for the Forest Cover Type dataset
columns = [
    'Elevation', 'Aspect', 'Slope',
    'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology',
    'Horizontal_Distance_To_Roadways', 'Hillshade_9am',
    'Hillshade_Noon', 'Hillshade_3pm',
    'Horizontal_Distance_To_Fire_Points'
]
# 4 wilderness area binary columns
columns += [f'Wilderness_Area_{i}' for i in range(1, 5)]
# 40 soil type binary columns
columns += [f'Soil_Type_{i}' for i in range(1, 41)]
columns += ['Cover_Type']  # Target

# Load dataset
df = pd.read_csv('covtype.data', header=None, names=columns)

print(f'Dataset Shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(f'Total Samples : {df.shape[0]:,}')
print(f'Total Features: {df.shape[1] - 1}')
print(f'Target Classes: {df["Cover_Type"].nunique()}')
print(f'\nMissing Values:')
print(df.isnull().sum().sum(), 'total missing values')
print(f'\nClass Distribution:')

class_names = {
    1: 'Spruce/Fir', 2: 'Lodgepole Pine', 3: 'Ponderosa Pine',
    4: 'Cottonwood/Willow', 5: 'Aspen', 6: 'Douglas-fir', 7: 'Krummholz'
}
class_dist = df['Cover_Type'].value_counts().sort_index()
for cls, count in class_dist.items():
    print(f'  Class {cls} ({class_names[cls]}): {count:,} samples')

In [ ]:
# Statistical summary of numerical features
df.describe()

## 📈 Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
labels = [class_names[i] for i in class_dist.index]
colors = ['#2ecc71','#3498db','#e74c3c','#f39c12','#9b59b6','#1abc9c','#e67e22']
axes[0].bar(labels, class_dist.values, color=colors, edgecolor='black')
axes[0].set_title('Vegetation Zone Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Vegetation Zone')
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', rotation=30)

# Pie chart
axes[1].pie(class_dist.values, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=140)
axes[1].set_title('Vegetation Zone Share (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class distribution plot saved!')

In [ ]:
# Distribution of key numerical features
numerical_cols = ['Elevation', 'Aspect', 'Slope',
                  'Horizontal_Distance_To_Hydrology',
                  'Horizontal_Distance_To_Roadways',
                  'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=40, color='#3498db', edgecolor='black', alpha=0.7)
    axes[i].set_title(col.replace('_', ' '), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Distribution of Numerical Features', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature distribution plot saved!')

In [ ]:
# Elevation vs Cover Type (boxplot)
plt.figure(figsize=(12, 5))
df_plot = df[['Elevation','Cover_Type']].copy()
df_plot['Vegetation Zone'] = df_plot['Cover_Type'].map(class_names)
df_plot.boxplot(column='Elevation', by='Vegetation Zone', figsize=(12,5),
                patch_artist=True)
plt.title('Elevation Distribution by Vegetation Zone', fontsize=13, fontweight='bold')
plt.suptitle('')
plt.xlabel('Vegetation Zone')
plt.ylabel('Elevation (meters)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('elevation_by_zone.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Elevation boxplot saved!')

In [ ]:
# Correlation heatmap (numerical features only)
plt.figure(figsize=(10, 8))
corr = df[numerical_cols + ['Cover_Type']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Correlation heatmap saved!')

## 🔧 Step 5 — Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Cover_Type', axis=1)
y = df['Cover_Type']

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape : {y.shape}')

In [ ]:
# ── Imputation ──
# (Dataset has no missing values, but we apply this as good practice)
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print('Before imputation — missing values:', X.isnull().sum().sum())
print('After  imputation — missing values:', X_imputed.isnull().sum().sum())
print('✅ Imputation complete!')

In [ ]:
# ── Variance Threshold (Feature Filtration) ──
print('Features before Variance Threshold:', X_imputed.shape[1])

selector = VarianceThreshold(threshold=0.01)
X_filtered = selector.fit_transform(X_imputed)
selected_features = X_imputed.columns[selector.get_support()].tolist()

print('Features after  Variance Threshold:', X_filtered.shape[1])
print(f'Removed {X_imputed.shape[1] - X_filtered.shape[1]} low-variance features')

# Convert back to DataFrame
X_filtered = pd.DataFrame(X_filtered, columns=selected_features)
print('\n✅ Variance threshold filtering done!')

In [ ]:
# ── Feature Scaling ──
# Scale only the continuous numerical features (not binary ones)
continuous_cols = [c for c in selected_features
                   if not c.startswith('Wilderness') and not c.startswith('Soil')]
binary_cols = [c for c in selected_features
               if c.startswith('Wilderness') or c.startswith('Soil')]

scaler = StandardScaler()
X_scaled_cont = pd.DataFrame(
    scaler.fit_transform(X_filtered[continuous_cols]),
    columns=continuous_cols
)
X_final = pd.concat(
    [X_scaled_cont, X_filtered[binary_cols].reset_index(drop=True)],
    axis=1
)

print(f'Final feature matrix shape: {X_final.shape}')
print('\n✅ Preprocessing complete! Summary:')
print(f'  → Imputation      : Mean imputation applied')
print(f'  → Variance Filter : threshold=0.01')
print(f'  → Scaling         : StandardScaler on {len(continuous_cols)} continuous features')

## ✂️ Step 6 — Train/Test Split

In [ ]:
# Use a subset for faster training (full dataset = 581K rows, takes long)
# Change sample_size to df.shape[0] for full dataset
SAMPLE_SIZE = 50000  # ~50k rows for demo; increase if you have time

df_sample = df.sample(n=SAMPLE_SIZE, random_state=42)
X_sample = X_final.loc[df_sample.index].reset_index(drop=True)
y_sample = df_sample['Cover_Type'].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

print(f'Training set  : {X_train.shape[0]:,} samples')
print(f'Test set      : {X_test.shape[0]:,} samples')
print(f'Features used : {X_train.shape[1]}')
print('\n✅ Train/Test split done (80/20)!')

## 🌲 Step 7 — Train Random Forest Model

In [ ]:
print('Training Random Forest Classifier...')
print('(This may take 1–2 minutes depending on Colab resources)\n')

rf_model = RandomForestClassifier(
    n_estimators=100,    # 100 decision trees
    max_depth=20,        # Max tree depth
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1            # Use all CPU cores
)

rf_model.fit(X_train, y_train)
print('✅ Model training complete!')

## 📊 Step 8 — Model Evaluation

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test)
train_acc = accuracy_score(y_train, rf_model.predict(X_train))
test_acc  = accuracy_score(y_test, y_pred)

print('=' * 50)
print('        MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'Training Accuracy : {train_acc*100:.2f}%')
print(f'Test Accuracy     : {test_acc*100:.2f}%')
print('=' * 50)
print()

# Classification report
target_names = [class_names[i] for i in sorted(y.unique())]
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues', xticks_rotation=30, ax=plt.gca())
plt.title('Confusion Matrix — Vegetation Zone Classification',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved!')

In [ ]:
# Feature Importance Plot
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({
    'Feature': X_final.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(12, 7))
colors_imp = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(feat_df))]
plt.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1],
         color=colors_imp[::-1], edgecolor='black')
plt.title('Top 20 Feature Importances — Random Forest',
          fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance plot saved!')

In [ ]:
# Per-class accuracy bar chart
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average=None, labels=sorted(y.unique())
)

metrics_df = pd.DataFrame({
    'Class': target_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1
})

x = np.arange(len(target_names))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width, precision, width, label='Precision', color='#3498db', edgecolor='black')
ax.bar(x,          recall,    width, label='Recall',    color='#2ecc71', edgecolor='black')
ax.bar(x + width,  f1,        width, label='F1-Score',  color='#e74c3c', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(target_names, rotation=30, ha='right')
ax.set_ylim(0, 1.1)
ax.set_title('Per-Class Precision, Recall & F1-Score', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
ax.axhline(y=test_acc, linestyle='--', color='gray', alpha=0.6, label=f'Overall Accuracy: {test_acc:.2f}')

plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Per-class metrics chart saved!')

## 🎯 Step 9 — Final Summary

In [ ]:
print('╔══════════════════════════════════════════════╗')
print('║   FOREST VEGETATION ZONE — PROJECT SUMMARY  ║')
print('╠══════════════════════════════════════════════╣')
print(f'║ Dataset     : Forest Cover Type (UCI/Kaggle) ║')
print(f'║ Total Rows  : 581,012                        ║')
print(f'║ Used Rows   : {SAMPLE_SIZE:,} (sampled)              ║')
print(f'║ Features    : {X_final.shape[1]} (after preprocessing)      ║')
print(f'║ Model       : Random Forest (100 trees)      ║')
print(f'║ Train Acc   : {train_acc*100:.2f}%                       ║')
print(f'║ Test Acc    : {test_acc*100:.2f}%                       ║')
print('╠══════════════════════════════════════════════╣')
print('║ Preprocessing Steps Applied:                 ║')
print('║   ✅ Pandas — data loading & EDA             ║')
print('║   ✅ SimpleImputer — mean imputation         ║')
print('║   ✅ VarianceThreshold — feature filtration  ║')
print('║   ✅ StandardScaler — feature scaling        ║')
print('║   ✅ Train/Test Split (80/20, stratified)    ║')
print('╚══════════════════════════════════════════════╝')

## 🔮 Step 10 — Predict on New Sample

In [ ]:
# Predict vegetation zone for a new forest area
# (Using first test sample as example)
sample = X_test.iloc[0:1]
predicted_class = rf_model.predict(sample)[0]
proba = rf_model.predict_proba(sample)[0]

print('🌲 Vegetation Zone Prediction for New Forest Area')
print('─' * 45)
print(f'Predicted Zone  : {class_names[predicted_class]} (Class {predicted_class})')
print(f'Actual Zone     : {class_names[y_test.iloc[0]]} (Class {y_test.iloc[0]})')
print()
print('Prediction Probabilities per Zone:')
for i, (cls, prob) in enumerate(zip(sorted(y.unique()), proba)):
    bar = '█' * int(prob * 30)
    print(f'  {class_names[cls]:<20} {bar} {prob*100:.1f}%')